# MADA - Fase 1: Tratamento dos logs

**Metodologia de Identificação de Viés Amplificado**

Este notebook contempla a fase de tratamento dos logs brutos extraídos da plataforma PCIbex Farm, usada para as anotações na fase empírica da metodologia.


### Conferencia do ambiente (Local x Colab)

In [18]:
import os
import sys

# 1. Detectar se o ambiente é o Google Colab
IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    print("Ambiente detectado: Google Colab")
    REPO_URL = "https://github.com/alexfeitosa72/MIVA.git"
    REPO_NAME = "MIVA"

    # Clonar apenas se a pasta ainda não existir no ambiente virtual do Colab
    if not os.path.exists(REPO_NAME):
        print(f"📥 Clonando {REPO_NAME}...")
        !git clone {REPO_URL}
    
    # Entrar na pasta do repositório no Colab
    if os.getcwd().split('/')[-1] != REPO_NAME:
        os.chdir(REPO_NAME)
    
    print(f"Diretório de trabalho ajustado: {os.getcwd()}")
else:
    print(f"Ambiente detectado: Local (VS Code/Jupyter)")
    print(f"Mantendo diretório atual: {os.getcwd()}")

# 2. Listar arquivos de forma compatível (funciona em Windows, Mac, Linux e Colab)
print("\nArquivos na pasta raiz:")
print(os.listdir('.'))

Ambiente detectado: Local (VS Code/Jupyter)
Mantendo diretório atual: b:\TechLab\MIVA

Arquivos na pasta raiz:
['.claude', '.git', '.lh', '.vscode', 'data', 'fase1_processamento_logs_pcibex copy.ipynb', 'fase1_processamento_logs_pcibex.ipynb', 'fase2_experimento_empirico copy.ipynb', 'fase2_experimento_empirico.ipynb', 'fase3_experimento_sintetico copy.ipynb', 'fase3_experimento_sintetico.ipynb', 'MIV.md', 'MIV2.md', 'resultados_MIV.ipynb', 'xtra_analise_qualitativa.ipynb', '_refs']


### 1. Setup e Leitura dos Dados
---

O dataset `MQD-1465` contém 1465 frases coletadas da plataforma "Meu Querido Diário" (www.meuqueridodiario.com.br)

INPUTS NECESSÁRIOS
* data/MQD-1465.csv (dataset original)
* data/logs_brutos (pasta com os logs brutos do experimento no PCIBex Farm)

TRATAMENTO PROPOSTO</br>
* Divisão do dataset em 10 blocos com no máximo 150 frases</br>
* Remoção de frases duplicadas no dataset original</br>
* Fixaçao de seed para reprodutibilidade




In [19]:
import pandas as pd
import numpy as np
import csv
from pathlib import Path
import random
import io

import warnings
warnings.filterwarnings('ignore')

In [20]:
# ============================================================================
# CONFIGURACAO
# ============================================================================
SEED = 42
INPUT_FILE = 'data/MQD-1465.csv'

print("=" * 80)
print("RANDOMIZACAO MQD-1465 (Seed: 42)")
print("=" * 80)

# ETAPA 1: CARREGAR E RANDOMIZAR
print("\n[1] Carregando e randomizando dados...")

df = pd.read_csv(INPUT_FILE, encoding='utf-8')
df.columns = ['id', 'frase', 'classificacao', 'juizes']
print(f"    {len(df)} registros carregados")

df_random = df.sample(frac=1, random_state=SEED).reset_index(drop=True)
print(f"    Randomizado com seed {SEED}")
print(f"    Primeiros IDs: {df_random['id'].head(5).tolist()}")

# ETAPA 2: REMOVER DUPLICATAS
print("\n[2] Removendo duplicatas...")

duplicatas = df_random[df_random.duplicated(subset=['frase'], keep=False)]
if len(duplicatas) > 0:
    print(f"    Duplicatas encontradas:")
    for frase, grupo in duplicatas.groupby('frase'):
        ids = grupo['id'].tolist()
        mantido = df_random[df_random['frase'] == frase]['id'].iloc[0]
        removidos = [i for i in ids if i != mantido]
        print(f"    - IDs {ids}: mantendo {mantido}, removendo {removidos}")

df_final = df_random.drop_duplicates(subset=['frase'], keep='first').reset_index(drop=True)
print(f"    Total apos remocao: {len(df_final)} registros")

# ETAPA 3: PADRONIZAR FRASES
print("\n[3] Padronizando frases...")
df_final['frase'] = df_final['frase'].str.strip().str.replace('"', '', regex=False)
print(f"    Espacos e aspas removidos")

# ETAPA 4: SALVAR ARQUIVO
num_registros = len(df_final)
OUTPUT_FILE = f'data/logs_processados/MQD-{num_registros}_random_sem_repeticao.csv'

print(f"\n[4] Salvando '{OUTPUT_FILE}'...")
os.makedirs(os.path.dirname(OUTPUT_FILE), exist_ok=True)

# Adiciona aspas nas frases antes de salvar
df_save = df_final.copy()
df_save['frase'] = '"' + df_save['frase'] + '"'
df_save.to_csv(OUTPUT_FILE, sep='\t', index=False, encoding='utf-8', quoting=csv.QUOTE_NONE, escapechar='\\')
print(f"    Arquivo salvo com sucesso!")

# ESTATISTICAS
print("\n" + "=" * 80)
print("ESTATISTICAS")
print("=" * 80)
print(f"\nTotal: {len(df_final)} registros\n")

print("Classificacao:")
for c in sorted(df_final['classificacao'].unique()):
    n = (df_final['classificacao'] == c).sum()
    print(f"  {c:>2}: {n:>4} ({100 * n / len(df_final):>5.1f}%)")

print("\nJuizes:")
for j in sorted(df_final['juizes'].unique()):
    n = (df_final['juizes'] == j).sum()
    print(f"  {j}: {n:>4} ({100 * n / len(df_final):>5.1f}%)")

print("\n" + "=" * 80)
print(f"CONCLUIDO! Arquivo: {OUTPUT_FILE} ({len(df_final)} registros)")
print("=" * 80)

RANDOMIZACAO MQD-1465 (Seed: 42)

[1] Carregando e randomizando dados...
    1465 registros carregados
    Randomizado com seed 42
    Primeiros IDs: [1039, 213, 314, 598, 931]

[2] Removendo duplicatas...
    Duplicatas encontradas:
    - IDs [475, 476]: mantendo 475, removendo [476]
    - IDs [952, 953]: mantendo 952, removendo [953]
    Total apos remocao: 1463 registros

[3] Padronizando frases...
    Espacos e aspas removidos

[4] Salvando 'data/logs_processados/MQD-1463_random_sem_repeticao.csv'...
    Arquivo salvo com sucesso!

ESTATISTICAS

Total: 1463 registros

Classificacao:
  -1:  544 ( 37.2%)
   0:  409 ( 28.0%)
   1:  510 ( 34.9%)

Juizes:
  2:  525 ( 35.9%)
  3:  936 ( 64.0%)
  4:    2 (  0.1%)

CONCLUIDO! Arquivo: data/logs_processados/MQD-1463_random_sem_repeticao.csv (1463 registros)


In [21]:
def processar_arquivo_randomizado():
    """
    Processa o arquivo randomizado:
    1. Renomeia coluna id para id_original
    2. Cria novo id sequencial
    3. Adiciona coluna de bloco (incremento a cada 150 registros)
    """
    arquivos = list(Path("data/logs_processados").glob("MQD-*_random_sem_repeticao.csv"))

    if not arquivos:
        raise FileNotFoundError("Nenhum arquivo MQD-*_random_sem_repeticao.csv encontrado!")

    if len(arquivos) > 1:
        print(f"AVISO: Multiplos arquivos encontrados. Usando o mais recente.")
        arquivos.sort(key=lambda p: p.stat().st_mtime, reverse=True)

    input_path = arquivos[0]
    print(f"Processando arquivo: {input_path}")

    df = pd.read_csv(
        input_path, sep='\t', usecols=['id', 'frase'],
        quoting=csv.QUOTE_ALL, quotechar='"', encoding='utf-8'
    )

    df = df.rename(columns={'id': 'id_original'})
    df.insert(0, 'id', range(1, len(df) + 1))
    df['bloco'] = (df.index // 150) + 1

    num_registros = len(df)
    output_path = f"data/logs_processados/MQD-{num_registros}_blocos.csv"

    df.to_csv(output_path, sep='\t', index=False, quoting=csv.QUOTE_ALL, quotechar='"')

    print(f"Arquivo salvo em: {output_path}")
    print(f"Total de registros: {len(df)} | Total de blocos: {df['bloco'].max()}")
    print(f"\nPrimeiras linhas:\n{df.head()}")

    return df

df_blocos_randomizados = processar_arquivo_randomizado()

Processando arquivo: data\logs_processados\MQD-1463_random_sem_repeticao.csv
Arquivo salvo em: data/logs_processados/MQD-1463_blocos.csv
Total de registros: 1463 | Total de blocos: 10

Primeiras linhas:
   id  id_original                                              frase  bloco
0   1         1039  Você sabia que o menino que mais vai te dar va...      1
1   2          213  Hoje com 21 anos, como que por auxilio lá do a...      1
2   3          314  Nesse momento to evitando ela, e ta me dando u...      1
3   4          598  Meus sentidos de garota diziam que tinha algum...      1
4   5          931  Aquela promessa que eu te fiz naquele natal, e...      1


### 2. Tratamento dos logs do PCIbex
---
* Criação dos blocos concatenados com todas as anotações
* Tamanho variável de blocos (anotadores x frases)
* Dataset com ParticipantMD5, GeneroCod, frase, Value e duracao

In [22]:
def carregar_blocos_randomizados():
    """Carrega o arquivo com as frases e seus respectivos blocos."""
    arquivos = list(Path("data/logs_processados").glob("MQD-*_blocos.csv"))

    if not arquivos:
        raise FileNotFoundError("Nenhum arquivo MQD-*_blocos.csv encontrado!")

    if len(arquivos) > 1:
        arquivos.sort(key=lambda p: p.stat().st_mtime, reverse=True)

    blocos_path = arquivos[0]
    print(f"Carregando arquivo: {blocos_path}")

    return pd.read_csv(
        blocos_path, sep='\t', quoting=csv.QUOTE_ALL,
        quotechar='"', encoding='utf-8'
    )


def extrair_genero(df):
    """Extrai o genero dos participantes do arquivo de log."""
    genero_data = df[
        (df["Label"] == "genero") &
        (df["PennElementName"] == "selecionaGenero") &
        (df["Parameter"] == "Selected")
    ][["ParticipantMD5", "Value"]].copy()

    genero_data["GeneroCod"] = genero_data["Value"].str.lower().map(
        lambda g: "m" if "masculino" in g else "f"
    )

    return genero_data[["ParticipantMD5", "GeneroCod"]]


def calcular_duracoes(df, num_bloco):
    """Calcula duracao entre Start e End para cada trial."""
    df_trials = df[
        (df["Label"] == "frases") &
        (df["Value"].isin(["Start", "End"]))
    ].copy()
    df_trials = df_trials.sort_values(["ParticipantMD5", "ItemNumber", "EventTime"])

    duracoes = []
    for (participant, item), group in df_trials.groupby(["ParticipantMD5", "ItemNumber"]):
        if len(group) != 2:
            continue
        start_time = group[group["Value"] == "Start"]["EventTime"].iloc[0]
        end_time = group[group["Value"] == "End"]["EventTime"].iloc[0]
        adjusted_item = item - 3 if num_bloco == 1 else item - 3 + ((num_bloco - 1) * 150)
        duracoes.append({
            "ParticipantMD5": participant,
            "ItemNumber": adjusted_item,
            "duracao": (end_time - start_time) / 1000.0
        })

    return pd.DataFrame(duracoes)


def processar_arquivo_log(file_path, frases_bloco, num_bloco):
    """Processa um arquivo de log correlacionando frases e calculando duracoes."""
    df = pd.read_csv(file_path, skiprows=19, header=None)
    df.columns = [
        "ReceptionTime", "ParticipantMD5", "Controller", "ItemNumber",
        "InnerElementNumber", "Label", "Group", "PennElementType",
        "PennElementName", "Parameter", "Value", "EventTime", "Comments"
    ]
    df["EventTime"] = pd.to_numeric(df["EventTime"], errors="coerce")

    generos = extrair_genero(df)
    df_duracoes = calcular_duracoes(df, num_bloco)

    df_classificacoes = df[
        (df["Label"] == "frases") &
        (df["Parameter"] == "Selection")
    ].copy()

    if num_bloco == 1:
        df_classificacoes["ItemNumber"] = df_classificacoes["ItemNumber"] - 3
    else:
        df_classificacoes["ItemNumber"] = df_classificacoes["ItemNumber"] - 3 + ((num_bloco - 1) * 150)

    df_final = (
        df_classificacoes
        .merge(frases_bloco[["id", "frase"]], left_on="ItemNumber", right_on="id", how="inner")
        .merge(generos, on="ParticipantMD5", how="left")
        .merge(df_duracoes, on=["ParticipantMD5", "ItemNumber"], how="left")
    )

    return df_final[["ParticipantMD5", "GeneroCod", "frase", "Value", "duracao"]]


def processar_todos_arquivos():
    """Processa todos os arquivos de log, correlacionando com os blocos corretos."""
    pasta_destino = "data/logs_processados"
    os.makedirs(pasta_destino, exist_ok=True)
    df_blocos = carregar_blocos_randomizados()

    for i in range(10):
        num_bloco = i + 1
        arquivo_origem = (
            "data/logs_brutos/results_prod.csv" if i == 0
            else f"data/logs_brutos/results_prod ({i}).csv"
        )
        arquivo_destino = f"{pasta_destino}/bloco_{num_bloco}_concatenado.csv"

        if not os.path.exists(arquivo_origem):
            print(f"Arquivo nao encontrado: {arquivo_origem}")
            continue

        print(f"\nProcessando bloco {num_bloco} - Arquivo: {arquivo_origem}")
        frases_bloco = df_blocos[df_blocos['bloco'] == num_bloco].copy()
        print(f"Frases no bloco {num_bloco}: {len(frases_bloco)}")

        try:
            df_processado = processar_arquivo_log(arquivo_origem, frases_bloco, num_bloco)

            if len(df_processado) > 0:
                df_processado.to_csv(
                    arquivo_destino, sep='\t', index=False,
                    quoting=csv.QUOTE_ALL, quotechar='"',
                    encoding='utf-8', float_format='%.3f'
                )
                print(f"Arquivo salvo: {arquivo_destino} ({len(df_processado)} classificacoes)")
            else:
                print(f"AVISO: Nenhuma classificacao encontrada para o bloco {num_bloco}")
        except Exception as e:
            print(f"Erro ao processar bloco {num_bloco}: {e}")

processar_todos_arquivos()

Carregando arquivo: data\logs_processados\MQD-1463_blocos.csv

Processando bloco 1 - Arquivo: data/logs_brutos/results_prod.csv
Frases no bloco 1: 150
Arquivo salvo: data/logs_processados/bloco_1_concatenado.csv (1350 classificacoes)

Processando bloco 2 - Arquivo: data/logs_brutos/results_prod (1).csv
Frases no bloco 2: 150
Arquivo salvo: data/logs_processados/bloco_2_concatenado.csv (1200 classificacoes)

Processando bloco 3 - Arquivo: data/logs_brutos/results_prod (2).csv
Frases no bloco 3: 150
Arquivo salvo: data/logs_processados/bloco_3_concatenado.csv (1800 classificacoes)

Processando bloco 4 - Arquivo: data/logs_brutos/results_prod (3).csv
Frases no bloco 4: 150
Arquivo salvo: data/logs_processados/bloco_4_concatenado.csv (1200 classificacoes)

Processando bloco 5 - Arquivo: data/logs_brutos/results_prod (4).csv
Frases no bloco 5: 150
Arquivo salvo: data/logs_processados/bloco_5_concatenado.csv (1350 classificacoes)

Processando bloco 6 - Arquivo: data/logs_brutos/results_prod 

### 

In [23]:
logs_path = Path('data/logs_processados')
bloco_files = list(logs_path.glob('bloco_*_concatenado.csv'))

dfs = [pd.read_csv(file, sep='\t') for file in bloco_files]
df_totais_com_duracao = pd.concat(dfs, ignore_index=True)

output_file = logs_path / 'MQD-13317_anotacoes_totais.csv'
df_totais_com_duracao.to_csv(output_file, sep='\t', index=False)

print(f"Arquivo criado: {output_file}")
print(f"Total de registros: {len(df_totais_com_duracao)}")

Arquivo criado: data\logs_processados\MQD-13317_anotacoes_totais.csv
Total de registros: 13317


In [24]:
# Separa por genero usando o DataFrame ja em memoria
df_totais_masculinos = df_totais_com_duracao[df_totais_com_duracao['GeneroCod'] == 'm']
df_totais_femininos = df_totais_com_duracao[df_totais_com_duracao['GeneroCod'] == 'f']

output_masculino = logs_path / 'MQD-7015_anotacoes_totais_masculinas.csv'
output_feminino = logs_path / 'MQD-6302_anotacoes_totais_femininas.csv'

df_totais_masculinos.to_csv(output_masculino, sep='\t', index=False)
df_totais_femininos.to_csv(output_feminino, sep='\t', index=False)

print(f"Masculino: {output_masculino} ({len(df_totais_masculinos)} registros)")
print(f"Feminino:  {output_feminino} ({len(df_totais_femininos)} registros)")

Masculino: data\logs_processados\MQD-7015_anotacoes_totais_masculinas.csv (7015 registros)
Feminino:  data\logs_processados\MQD-6302_anotacoes_totais_femininas.csv (6302 registros)


## Geração do Dataset Final: 4 Classificações com Maioria Simples
Esta seção gera o dataset final garantindo:
1. **Exatamente 4 classificações** por gênero para cada frase
2. **Maioria Simples** definido (sem empate 2-2)
3. **Paridade** entre gêneros (apenas frases presentes em ambos)

**Lógica de seleção:**
- Usar as 4 primeiras anotações (ordem temporal)
- Se houver empate 2-2, substituir uma das classificações pela próxima, buscando desempatar
- Se impossível resolver empate, descartar a frase

**Maioria válida** (classe com mais votos, sem empate no topo):
- ✓ 4-0-0 (unanimidade)
- ✓ 3-1-0 
- ✓ 2-1-1 (uma classe à frente das demais)
- ✗ 2-2-0 (empate — descartada)

In [25]:
# =============================================================================
# GERACAO DO DATASET FINAL: 4 CLASSIFICACOES COM MAIORIA SIMPLES
# =============================================================================

from itertools import combinations

df_masc_raw = pd.read_csv(logs_path / 'MQD-7015_anotacoes_totais_masculinas.csv', sep='\t')
df_fem_raw = pd.read_csv(logs_path / 'MQD-6302_anotacoes_totais_femininas.csv', sep='\t')

df_masc_raw['ordem_original'] = df_masc_raw.index
df_fem_raw['ordem_original'] = df_fem_raw.index

print('=' * 80)
print('GERACAO DO DATASET FINAL')
print('=' * 80)
print(f'Anotacoes masculinas: {len(df_masc_raw)}')
print(f'Anotacoes femininas: {len(df_fem_raw)}')


def tem_pluralidade(contagem):
    valores = sorted(contagem.values, reverse=True)
    return not (len(valores) >= 2 and valores[0] == valores[1])


def selecionar_4_com_pluralidade(grupo):
    # Ordena por posicao original no arquivo (registros mais antigos primeiro).
    # Itera sobre combinacoes de 4 em ordem lexicografica, garantindo
    # que a primeira combinacao valida use os registros mais antigos possiveis.
    grupo = grupo.sort_values('ordem_original').reset_index(drop=True)
    if len(grupo) < 4:
        return None
    for idxs in combinations(range(len(grupo)), 4):
        selecao = grupo.iloc[list(idxs)]
        if tem_pluralidade(selecao['Value'].value_counts()):
            return selecao
    return None


def processar_grupo(df, genero):
    resultados = []
    descartadas = []
    for frase in df['frase'].unique():
        grupo = df[df['frase'] == frase]
        selecao = selecionar_4_com_pluralidade(grupo)
        if selecao is None:
            descartadas.append(frase)
            continue
        contagem = selecao['Value'].value_counts()
        resultados.append({
            'frase': frase,
            f'total_classificacoes_{genero}': 4,
            f'classificacao_majoritaria_{genero}': contagem.idxmax(),
            f'votos_maioria_{genero}': contagem.max(),
            f'total_positiva_{genero}': (selecao['Value'] == 'positiva').sum(),
            f'total_negativa_{genero}': (selecao['Value'] == 'negativa').sum(),
            f'total_neutra_{genero}': (selecao['Value'] == 'neutra').sum(),
            f'duracao_media_{genero}': selecao['duracao'].mean()
        })
    return pd.DataFrame(resultados), descartadas


print('\n[1] Processando anotacoes masculinas...')
df_result_masc, descartadas_masc = processar_grupo(df_masc_raw, 'masculino')
print(f'    OK {len(df_result_masc)} frases com maioria simples valida')
print(f'    X  {len(descartadas_masc)} frases descartadas (empate irresolvivel)')

print('\n[2] Processando anotacoes femininas...')
df_result_fem, descartadas_fem = processar_grupo(df_fem_raw, 'feminino')
print(f'    OK {len(df_result_fem)} frases com maioria simples valida')
print(f'    X  {len(descartadas_fem)} frases descartadas (empate irresolvivel)')

print('\n[3] Combinando datasets...')
df_final = df_result_masc.merge(df_result_fem, on='frase', how='inner')
print(f'    {len(df_final)} frases em comum (ambos generos com maioria simples valida)')

df_final['concordancia_grupos'] = (
    df_final['classificacao_majoritaria_masculino'] ==
    df_final['classificacao_majoritaria_feminino']
).astype(int)

n_concordam = df_final['concordancia_grupos'].sum()
n_discordam = len(df_final) - n_concordam

print('\n[4] Concordancia entre grupos:')
if len(df_final) > 0:
    print(f'    Concordam: {n_concordam} ({100 * n_concordam / len(df_final):.1f}%)')
    print(f'    Discordam: {n_discordam} ({100 * n_discordam / len(df_final):.1f}%)')
else:
    print('    Sem dados para comparar')

colunas_ordem = [
    'frase',
    'duracao_media_masculino', 'total_classificacoes_masculino',
    'classificacao_majoritaria_masculino', 'votos_maioria_masculino',
    'total_positiva_masculino', 'total_negativa_masculino', 'total_neutra_masculino',
    'duracao_media_feminino', 'total_classificacoes_feminino',
    'classificacao_majoritaria_feminino', 'votos_maioria_feminino',
    'total_positiva_feminino', 'total_negativa_feminino', 'total_neutra_feminino',
    'concordancia_grupos'
]
df_final = df_final[colunas_ordem]

output_file = logs_path / f'MQD-{len(df_final)}_majoritarias.csv'
df_final.to_csv(output_file, sep='\t', index=False)
print(f'\n[5] Arquivo salvo: {output_file.name}')

print('\n' + '=' * 80)
print('ESTATISTICAS DO DATASET FINAL')
print('=' * 80)
print('\nDistribuicao de votos na maioria:')
print('\nMasculino:')
print(df_final['votos_maioria_masculino'].value_counts().sort_index())
print('\nFeminino:')
print(df_final['votos_maioria_feminino'].value_counts().sort_index())
print('\nDistribuicao de classes majoritarias:')
print('\nMasculino:')
print(df_final['classificacao_majoritaria_masculino'].value_counts())
print('\nFeminino:')
print(df_final['classificacao_majoritaria_feminino'].value_counts())
print('\n' + '=' * 80)
print('DATASET FINAL GERADO COM SUCESSO')
print('=' * 80)

GERACAO DO DATASET FINAL
Anotacoes masculinas: 7015
Anotacoes femininas: 6302

[1] Processando anotacoes masculinas...
    OK 1393 frases com maioria simples valida
    X  70 frases descartadas (empate irresolvivel)

[2] Processando anotacoes femininas...
    OK 1273 frases com maioria simples valida
    X  190 frases descartadas (empate irresolvivel)

[3] Combinando datasets...
    1222 frases em comum (ambos generos com maioria simples valida)

[4] Concordancia entre grupos:
    Concordam: 1033 (84.5%)
    Discordam: 189 (15.5%)

[5] Arquivo salvo: MQD-1222_majoritarias.csv

ESTATISTICAS DO DATASET FINAL

Distribuicao de votos na maioria:

Masculino:
votos_maioria_masculino
2    129
3    469
4    624
Name: count, dtype: int64

Feminino:
votos_maioria_feminino
2    115
3    610
4    497
Name: count, dtype: int64

Distribuicao de classes majoritarias:

Masculino:
classificacao_majoritaria_masculino
positiva    481
negativa    450
neutra      291
Name: count, dtype: int64

Feminino:
cla